[![Roboflow Notebooks](https://media.roboflow.com/notebooks/template/bannertest2-2.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672932710194)](https://github.com/roboflow/notebooks)

# How to Tune Parameters to your Tracker
In this notebook you will download MOT17 dataset, format it for using the Tuner class, and tune the hiperparameters of the tracker.
Finally evaluate the best parameters found over the evaluation set.  

## Install Trackers


In [1]:
!pip install trackers
!pip install trackers[tune]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.4/217.4 kB 9.7 MB/s eta 0:00:00


## Download the dataset
`Trackers` library provides us with CLI and python code to download the main datasets used in Multi-Object Tracking

In [2]:
!trackers download --help

usage: trackers download [-h] [--list] [--split SPLIT] [--asset ASSET]
                         [-o OUTPUT] [--cache-dir CACHE_DIR]
                         [dataset]

Download tracking datasets from the official trackers bucket.

positional arguments:
  dataset               Dataset name (e.g. mot17, sportsmot).

options:
  -h, --help            show this help message and exit
  --list                List available datasets, splits, and asset types.
  --split SPLIT         Comma-separated splits to download (e.g.
                        train,val,test). If omitted, all available splits are
                        downloaded.
  --asset ASSET         Comma-separated assets to download:
                        annotations,frames,detections. If omitted, all
                        available assets are downloaded.
  -o OUTPUT, --output OUTPUT
                        Output directory (default: current directory).
  --cache-dir CACHE_DIR
                        Cache directory for downloaded

In [3]:
!trackers download mot17 --split train,val --asset detections,annotations

[download] mot17:train:detections
mot17-train-public-detections.zip ━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.5 MB/s
[extract] mot17:train:detections
[done] mot17:train:detections
[download] mot17:train:annotations
mot17-train-annotations.zip ━━━━━━━━━━━━━━━━━━━━━━━━━━━ 694.6/694.6 kB 33.7 MB/s
[extract] mot17:train:annotations
[done] mot17:train:annotations
[download] mot17:val:detections
mot17-val-public-detections.zip ━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 59.9 MB/s
[extract] mot17:val:detections
[done] mot17:val:detections
[download] mot17:val:annotations
mot17-val-annotations.zip ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.2/615.2 kB 63.5 MB/s
[extract] mot17:val:annotations
[done] mot17:val:annotations


## Format dataset for tuning

Tuner class expects a directory for track sand a directory for detections like:
```text
data/
├── gt/
│   ├── MOT17-02-FRCNN
│   │   └── gt
│   │       └── gt.txt
│   │
│   ├── MOT17-04-FRCNN
│   └── ...
└── detections/
    ├── MOT17-02-FRCNN.txt
    ├── MOT17-04-FRCNN.txt
    └── ...
```
So lets format the downloaded dataset to fit this requirement

In [ ]:
import os
import shutil

gt_base_dir = "mot17/"
train_split_gt_base = os.path.join(gt_base_dir, "train")

# Flatten detection files for 'train' split
formatted_det_output_dir = os.path.join(train_split_gt_base, "det_flattened")
os.makedirs(formatted_det_output_dir, exist_ok=True)

for seq_folder in sorted(os.listdir(train_split_gt_base)):
    full_seq_path = os.path.join(train_split_gt_base, seq_folder)
    if os.path.isdir(full_seq_path):
        source_det_path = os.path.join(full_seq_path, "det", "det.txt")
        destination_det_path = os.path.join(formatted_det_output_dir, f"{seq_folder}.txt")
        if os.path.exists(source_det_path):
            shutil.copy(source_det_path, destination_det_path)
            print(f"Copied Det {source_det_path} to {destination_det_path}")

print("Detection files flattening complete.")

# Define variables for the tuner
det_dir = formatted_det_output_dir
print(f"det_dir set to: {det_dir}")

Copied Det /content/mot17/train/MOT17-02-DPM/det/det.txt to /content/mot17/train/det_flattened/MOT17-02-DPM.txt
Copied Det /content/mot17/train/MOT17-02-FRCNN/det/det.txt to /content/mot17/train/det_flattened/MOT17-02-FRCNN.txt
Copied Det /content/mot17/train/MOT17-02-SDP/det/det.txt to /content/mot17/train/det_flattened/MOT17-02-SDP.txt
Copied Det /content/mot17/train/MOT17-04-DPM/det/det.txt to /content/mot17/train/det_flattened/MOT17-04-DPM.txt
Copied Det /content/mot17/train/MOT17-04-FRCNN/det/det.txt to /content/mot17/train/det_flattened/MOT17-04-FRCNN.txt
Copied Det /content/mot17/train/MOT17-04-SDP/det/det.txt to /content/mot17/train/det_flattened/MOT17-04-SDP.txt
Copied Det /content/mot17/train/MOT17-05-DPM/det/det.txt to /content/mot17/train/det_flattened/MOT17-05-DPM.txt
Copied Det /content/mot17/train/MOT17-05-FRCNN/det/det.txt to /content/mot17/train/det_flattened/MOT17-05-FRCNN.txt
Copied Det /content/mot17/train/MOT17-05-SDP/det/det.txt to /content/mot17/train/det_flatten

!! This next cells will be removed once everything is merged

In [5]:
!pip uninstall trackers

Found existing installation: trackers 2.3.0
Uninstalling trackers-2.3.0:
  Would remove:
    /usr/local/bin/trackers
    /usr/local/lib/python3.12/dist-packages/trackers-2.3.0.dist-info/*
    /usr/local/lib/python3.12/dist-packages/trackers/*
Proceed (Y/n)? y
  Successfully uninstalled trackers-2.3.0


In [6]:
!pip install -q git+https://github.com/omkar-334/trackers.git@feat/cli/tune-stage2 --no-cache

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [7]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 6.0 MB/s eta 0:00:00


## Tune your favorite tracker

In [8]:
from trackers.tune import Tuner

Here we define the tuner, we select:
 - The `SORT` tracker.
 - `gt_dir`: path to ground truth tracks.
 - `det_dir`: path to detections over which the tracker will act.
 -  `objective`: objective metric to maximize.
 - `metrics`: list of all the metrics we want to keep a track of.
 - `n_trials`: number of combinations of hyperparameters to try.  

In [ ]:
gt_dir = 'mot17/train'

tuner = Tuner("sort", gt_dir, det_dir,
              objective = "HOTA",
              metrics=["HOTA"],
              n_trials = 15)

### Run the parameter tuning
This will take a while: the tuner is trying 15 different combinations of parameters, finding by itself which changes to make to the parameters in order to maximize our objetive metric

In [18]:
best_params = tuner.run()

[I 2026-03-27 13:37:17,698] A new study created in memory with name: trackers-tune-sort
[I 2026-03-27 13:37:28,929] Trial 0 finished with value: 0.13256008568177796 and parameters: {'lost_track_buffer': 39, 'track_activation_threshold': 0.2377381638424672, 'minimum_consecutive_frames': 3, 'minimum_iou_threshold': 0.2845933698271799}. Best is trial 0 with value: 0.13256008568177796.
[I 2026-03-27 13:37:40,198] Trial 1 finished with value: 0.13523735123132224 and parameters: {'lost_track_buffer': 56, 'track_activation_threshold': 0.6296591435085503, 'minimum_consecutive_frames': 2, 'minimum_iou_threshold': 0.0548526844745505}. Best is trial 1 with value: 0.13523735123132224.
[I 2026-03-27 13:37:52,544] Trial 2 finished with value: 0.13291667591600373 and parameters: {'lost_track_buffer': 80, 'track_activation_threshold': 0.8083421544621436, 'minimum_consecutive_frames': 2, 'minimum_iou_threshold': 0.4882273002180623}. Best is trial 1 with value: 0.13523735123132224.
[I 2026-03-27 13:38:0

and the best parameters are ...

In [19]:
print(best_params)

{'lost_track_buffer': 69, 'track_activation_threshold': 0.3485735415165917, 'minimum_consecutive_frames': 4, 'minimum_iou_threshold': 0.06322898044739839}


## Evaluate over the other set

First we will import some utilities to evaluate.

In [20]:
from trackers.tune.tuner import _run_tracker_on_detections
from trackers.eval.evaluate import evaluate_mot_sequences
from trackers import SORTTracker
from pathlib import Path
import tempfile

We will now flatten the validation detections to: `/content/mot17/val/det_flattened`


In [13]:
val_split_gt_base = os.path.join(gt_base_dir, "val")

formatted_det_output_dir = os.path.join(val_split_gt_base, "det_flattened")
os.makedirs(formatted_det_output_dir, exist_ok=True)

for seq_folder in sorted(os.listdir(val_split_gt_base)):
    full_seq_path = os.path.join(val_split_gt_base, seq_folder)
    if os.path.isdir(full_seq_path):
        source_det_path = os.path.join(full_seq_path, "det", "det.txt")
        destination_det_path = os.path.join(formatted_det_output_dir, f"{seq_folder}.txt")
        if os.path.exists(source_det_path):
            shutil.copy(source_det_path, destination_det_path)
            print(f"Copied Det {source_det_path} to {destination_det_path}")
        else:
            print(f"Warning: Det {source_det_path} not found.")

print("Detection files flattening complete.")


Copied Det /content/mot17/val/MOT17-02-DPM/det/det.txt to /content/mot17/val/det_flattened/MOT17-02-DPM.txt
Copied Det /content/mot17/val/MOT17-02-FRCNN/det/det.txt to /content/mot17/val/det_flattened/MOT17-02-FRCNN.txt
Copied Det /content/mot17/val/MOT17-02-SDP/det/det.txt to /content/mot17/val/det_flattened/MOT17-02-SDP.txt
Copied Det /content/mot17/val/MOT17-04-DPM/det/det.txt to /content/mot17/val/det_flattened/MOT17-04-DPM.txt
Copied Det /content/mot17/val/MOT17-04-FRCNN/det/det.txt to /content/mot17/val/det_flattened/MOT17-04-FRCNN.txt
Copied Det /content/mot17/val/MOT17-04-SDP/det/det.txt to /content/mot17/val/det_flattened/MOT17-04-SDP.txt
Copied Det /content/mot17/val/MOT17-05-DPM/det/det.txt to /content/mot17/val/det_flattened/MOT17-05-DPM.txt
Copied Det /content/mot17/val/MOT17-05-FRCNN/det/det.txt to /content/mot17/val/det_flattened/MOT17-05-FRCNN.txt
Copied Det /content/mot17/val/MOT17-05-SDP/det/det.txt to /content/mot17/val/det_flattened/MOT17-05-SDP.txt
Copied Det /cont

And now we can run the tracking over the validation set and then see the metrics

In [ ]:
gt_dir = Path("mot17/val/")
det_dir = Path("mot17/val/det_flattened/")
sequences = sorted(p.stem for p in det_dir.glob("*.txt"))
eval_kwargs = dict(
    gt_dir=gt_dir,
    metrics=["CLEAR", "HOTA", "Identity"],
)

def run_eval(tracker):
    with tempfile.TemporaryDirectory() as tmp:
        pred_dir = Path(tmp)
        for seq in sequences:
            tracker.reset()
            _run_tracker_on_detections(tracker, det_dir / f"{seq}.txt", pred_dir / f"{seq}.txt")
        return evaluate_mot_sequences(tracker_dir=pred_dir, **eval_kwargs)


First we will evaluate with default parameters

In [24]:
result_tuned = run_eval(SORTTracker())
print(result_tuned.table(columns=["HOTA", "MOTA" , "IDF1"]))

Sequence                        HOTA    MOTA    IDF1
----------------------------------------------------
MOT17-02-FRCNN                33.742  30.030  36.215
MOT17-04-FRCNN                55.348  48.813  62.383
MOT17-05-FRCNN                46.600  50.640  58.909
MOT17-09-FRCNN                49.461  50.747  56.474
MOT17-10-FRCNN                49.367  51.528  54.207
MOT17-11-FRCNN                48.992  54.572  54.689
MOT17-13-FRCNN                54.663  55.672  65.016
----------------------------------------------------
COMBINED                      49.950  46.769  56.088


and now lets see how tuning parameters worked out

In [23]:
result_tuned = run_eval(SORTTracker(**best_params))

print(result_tuned.table(columns=["HOTA", "MOTA", "IDF1"]))

Sequence                        HOTA    MOTA    IDF1
----------------------------------------------------
MOT17-02-FRCNN                34.437  29.646  36.246
MOT17-04-FRCNN                55.560  48.805  62.519
MOT17-05-FRCNN                45.407  50.164  56.731
MOT17-09-FRCNN                49.259  50.226  56.254
MOT17-10-FRCNN                49.032  50.650  54.657
MOT17-11-FRCNN                48.085  54.240  54.501
MOT17-13-FRCNN                57.243  58.460  69.077
----------------------------------------------------
COMBINED                      50.124  46.677  56.316


As you can see, we maximized for HOTA and got +0.2% HOTA, -0.1% MOTA and +0.2% IDF1. And by increasing the number of trials we can have even bigger improvements!

Try it out and get your Tracker to the best performance!

**Note**: MOT17 provides detections coming from different object detectors, we evaluate our trackers using the ones from FRCNN.